In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

In [2]:
train_df = pd.read_csv('../dataset/train.csv')
test_df = pd.read_csv('../dataset/test.csv')

In [8]:
train_df.info(), test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  float64
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  object 
 11  time_str       77299 non-null  object 
 12  is_train       77299 non-null  int64  
dtypes: float64(3), int64(4), object(6)
memory usage: 7.7+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------     

(None, None)

###  Fix the Train/Test Timestamp Bug

In [10]:
# Converts training floats (e.g., 2.15 -> "02:15", 2.3 -> "02:30") to matching string formats
def convert_train_timestamp(x):
    hour = int(x)
    minute = int(round((x - hour) * 100))
    return f"{hour:02d}:{minute:02d}"

print(convert_train_timestamp(2.15))  # Output: "02:15"

02:15


In [9]:
import test


train_df['time_str'] = train_df['timestamp'].apply(convert_train_timestamp)
test_df['time_str'] = test_df['timestamp'].astype(str)
train_df['time_str'].head(), test_df['time_str'].head()

(0    00:00
 1    00:00
 2    00:00
 3    00:00
 4    00:00
 Name: time_str, dtype: object,
 0    2:15
 1    2:15
 2    2:15
 3    2:15
 4    2:15
 Name: time_str, dtype: object)

In [6]:
# Tag files before concatenating for clean preprocessing
train_df['is_train'] = 1
test_df['is_train'] = 0
test_df['demand'] = np.nan

In [ ]:
full_df = pd.concat([train_df, test_df], ignore_index=True)

In [11]:
# Generate uniform datetime features
full_df['dt'] = pd.to_datetime(full_df['time_str'], format='%H:%M')

full_df['hour'] = full_df['dt'].dt.hour
full_df['minute'] = full_df['dt'].dt.minute
full_df['time_in_minutes'] = full_df['hour'] * 60 + full_df['minute']


In [12]:
# Drop raw and transitional string columns
full_df.drop(columns=['timestamp', 'time_str', 'dt'], inplace=True)


### Treat Null Value

In [ ]:
full_df.isnull().sum()

In [13]:
# Clean Missing Values
full_df['Temperature'] = full_df['Temperature'].fillna(full_df['Temperature'].median())

full_df['RoadType'] = full_df['RoadType'].fillna('Unknown')
full_df['Weather'] = full_df['Weather'].fillna('Unknown')

### Feature Engineering

In [14]:
# Treat the day feature as a binary target day flag
full_df['is_day_49'] = (full_df['day'] == 49).astype(int)

# Spatial Granularity: Group geohashes into broader neighborhood zones
full_df['geohash_zone_4'] = full_df['geohash'].str[:4]
full_df['geohash_zone_5'] = full_df['geohash'].str[:5]

In [15]:
# Historical Baseline calculated strictly from training rows to prevent leakage
train_only = full_df[full_df['is_train'] == 1]
global_mean_demand = train_only['demand'].mean()

In [16]:
# Map historical average demand per precise geohash location
geohash_maps = train_only.groupby('geohash')['demand'].mean().to_dict()
full_df['geohash_avg_demand'] = full_df['geohash'].map(geohash_maps).fillna(global_mean_demand)

In [17]:
# Map historical average demand per broader regional zone
zone4_maps = train_only.groupby('geohash_zone_4')['demand'].mean().to_dict()
full_df['zone4_avg_demand'] = full_df['geohash_zone_4'].map(zone4_maps).fillna(global_mean_demand)

#### Type casting natively

In [19]:
full_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119077 entries, 0 to 119076
Data columns (total 19 columns):
 #   Column              Non-Null Count   Dtype   
---  ------              --------------   -----   
 0   Index               119077 non-null  int64   
 1   geohash             119077 non-null  category
 2   day                 119077 non-null  int64   
 3   demand              77299 non-null   float64 
 4   RoadType            119077 non-null  category
 5   NumberofLanes       119077 non-null  int64   
 6   LargeVehicles       119077 non-null  category
 7   Landmarks           119077 non-null  category
 8   Temperature         119077 non-null  float64 
 9   Weather             119077 non-null  category
 10  is_train            119077 non-null  int64   
 11  hour                119077 non-null  int32   
 12  minute              119077 non-null  int32   
 13  time_in_minutes     119077 non-null  int32   
 14  is_day_49           119077 non-null  int64   
 15  geohash_zone_4   

In [18]:
categorical_cols = ['geohash', 'geohash_zone_4', 'geohash_zone_5', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
for col in categorical_cols:
    full_df[col] = full_df[col].astype('category')

In [20]:
# Split Back into Train and Test Sets
train_clean = full_df[full_df['is_train'] == 1].drop(columns=['is_train', 'Index'])
test_clean = full_df[full_df['is_train'] == 0].drop(columns=['is_train', 'demand'])
test_indices = full_df[full_df['is_train'] == 0]['Index']

In [22]:
X = train_clean.drop(columns=['demand'])
y = train_clean['demand']
X.columns

Index(['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'Weather', 'hour', 'minute',
       'time_in_minutes', 'is_day_49', 'geohash_zone_4', 'geohash_zone_5',
       'geohash_avg_demand', 'zone4_avg_demand'],
      dtype='object')

### Train

In [23]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:
model = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.85,            # Sample 85% rows per tree to guard against variance
    colsample_bytree=0.85,     # Sample 85% features per tree 
    tree_method='hist',
    enable_categorical=True,
    early_stopping_rounds=50,
    random_state=42
)

In [25]:
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)


[0]	validation_0-rmse:0.13844
[100]	validation_0-rmse:0.03282
[200]	validation_0-rmse:0.02954
[300]	validation_0-rmse:0.02874
[400]	validation_0-rmse:0.02837
[500]	validation_0-rmse:0.02816
[600]	validation_0-rmse:0.02803
[700]	validation_0-rmse:0.02794
[800]	validation_0-rmse:0.02788
[900]	validation_0-rmse:0.02780
[1000]	validation_0-rmse:0.02775
[1100]	validation_0-rmse:0.02774
[1109]	validation_0-rmse:0.02774


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.85
,device,None
,early_stopping_rounds,50
,enable_categorical,True
,eval_metric,None


### Evaluate

In [26]:
val_preds = model.predict(X_val)
print(f"\nFinal Validation R2 Score: {r2_score(y_val, val_preds):.4f}")


Final Validation R2 Score: 0.9610


### Generate Submission

In [27]:
test_features = test_clean.drop(columns=['Index'])
test_preds = model.predict(test_features)

submission = pd.DataFrame({
    'Index': test_indices,
    'demand': test_preds
})

In [ ]:
submission.to_csv('c_submission.csv', index=False)
print("Optimized submission generated successfully!")

Optimized submission generated successfully!
